# CE310 Class 04 

_Real Valued Genetic Algorithms and Travelling Sales-Person Problem (TSP)_

*Traveling Salesperson Problem:* Given  ```N``` cities with mutual distance ```d[i,j]```, find the order in which to visit the cities and get back to the city you started from, _as to minimise the distance travelled_. 

While solutions to the TSP are _tours_, they can easily be represented as _permutations_. For instance, ```[4, 3, 1, 2]``` means "visit city 4, then 3, then 1, then 2 and then go back to 4 to complete the tour". So, the total distance travelled would be: ```d[4,3] + d[3,1] + d[1,2] + d[2,4]``` 

_TASKS:_ 

1. Understand and test the following code that generates a random set of ```N``` cities and precomputes an array of distances ```d[i,j]``` for all pairs of cities.

```python
import numpy as np

# Set the number of cities
N = 20

# Generate an array random x and y coordinates for the cities
# in a region of 100 by 100 miles
xy = 100 * np.random.rand(N,2) 

# Create an numpy array with N rows and N columns for storing 
# the distances between every pair of city

d = np.empty((N,N))
for i in range(N):
    for j in range(N):
        # Calculate Euclidean distance
        dx = xy[i,0] - xy[j,0]
        dy = xy[i,1] - xy[j,1]
        d[i,j] = np.sqrt(dx**2 + dy**2)
```
You can use the following code to plot the cities and visualise the distance matrix:
```python
# Visualising the distance matrix as a table
import pandas as pd
print(pd.DataFrame(d))

# Visualising the distance matrix as a heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(d, annot=False, cmap='viridis')
plt.title("Distance Matrix as Heatmap")
plt.ylabel("From City")
plt.xlabel("To City")
plt.show()
```

2. Study and test the following TSP simulator, that uses the array ```d``` to calculate the total distance travelled by a salesperson if visiting the cities in the order specified by a particular ```permutation```:

```Python
def tsp_simulator(permutation):
    dist = 0
    for city in range(N-1):
        dist += d[permutation[city],permutation[city+1]]
    dist += d[permutation[0],permutation[city+1]] # Adding the return leg to the starting city
    return dist
```

3. Like with did with the bin-packing problem, let us use *random search* to solve the TSP. Specifically, generate 1,000,000 random permutations (i.e., random tours), running the TSP simulator on each and ensuring you save the best permutation found.

  We have already seen a couple of ways of generating random permutations. Here is another one:

```Python
permutation = np.random.permutation(range(N))
```
   * What's the minimal tour length you can get by random search?
   * Visualise your best tour using the following code
```python
# Plotting tours
def plot_tour(perm):
    for city in range(N-1):
        plt.plot([xy[perm[city]][0],xy[perm[city+1]][0]],
                 [xy[perm[city]][1],xy[perm[city+1]][1]],'b-')
    plt.plot([xy[perm[0]][0],xy[perm[city+1]][0]],
             [xy[perm[0]][1],xy[perm[city+1]][1]],'b-')
    plt.plot(xy[:,0],xy[:,1],'r*')
    plt.show()
```
   * See how the distance matrix changes if you reordered the cities based on your best permutation. The calculation is trivial: ```permuted_d = d[best_perm][:, best_perm]```. Visualise ```permuted_d``` and compare with the original. Can you explain what you see? 

4. Now let us transition to a _new way of representing and generating random permutations which will make it easy for us to use a GA to solve TSPs_.

 * Imagine that we _assign to each city a numerical value representing the **priority**_ with which it should be visited (and assume there are no ties).
 * It is then easy to turn these priorities into a permutation/tour: the city with highest priority should be visited first, the city with the second highest priority should be second, and so on.
 * So, an _array of floats in an indirect representation for permutations_.
 * How do we transform a ```priorities``` array into a ```permutation```? Super easy:
```python
permutation = np.argsort(priorities)[::-1]
```
Test what happens if you apply ```argsort``` to the list ```[0.4, 0.2, 1.1, 0.1]```. 

5. Test that this new representation works OK, by modifying your random search code developed in Task 3 above so that it generates permutations as follows:
```python
priorities = np.random.rand(N) # An array of random numbers between 0 and 1
permutation = np.argsort(priorities)[::-1] 
```
Run the new random search code, and see if you more or less get the same result as in Task 3.

6. The plan is to use a _Real-Valued GA_ to solve the TSP problem. To do this we need to **copy over and adapt the binary GA we defined in last week's lab** so it works on real-valued chromosomes. If you didn't complete the implementation of the binary GA, see my solutions now available in Moodle here [https://moodle.essex.ac.uk/mod/folder/view.php?id=1418183].

To get started, let us just do minimal changes to it so we get something running as soon as possible:

   * Change the _initisation_ so that we create a population of _arrays of floats_ (priorities) (using ```np.random.rand(N)```) instead of 0s and 1s 
   * Redefine the _fitness function_ as follows:
```python
def fitness(priorities):
    permutation = np.argsort(priorities)[::-1]
    distance = tsp_simulator(permutation)
    fitness = 1.0 / distance # So the higher the distance the worse the fitness
    return fitness
```
   * For now, let us _switch off mutation_ by either commenting out any use of ```mutation()``` in the code or redefining it as
```python
# A fake mutation which just CLONES a parent
def mutate():
    idx = tournament()
    parent = population[idx]
    return parent
```
Later we will redefine it properly.

   * Also, for now, leave  ```crossover()``` as it is. We know that one-point crossover is not very suitable for floats, but we will fix this later.

7. Run this GA for 1,000,000 iterations (fitness evaluations) to see if the results we obtain are comparable with those obtained by random search. 
   
8. Let us now _reintroduce mutation_. One simple way to mutate is to add small random uniform displacements. For instance:
```python
def mutate():
    idx = tournament()
    parent = population[idx]
    offset = 0.05 * (np.random.rand(N) - 0.5)
    offspring = parent + offset
    return offspring
```
* Note that 0.05 is not necessarily optimal, so some tweaking may be necesary.

* A more sophisticated method would be to use normal deviates with 0 mean and standard deviation 0.05. See Unit 3 lecture slides 32 and 33. 

* _Re-run the algorithm with this mutation included_. Do you get better results (numerically, plotting the tours and looking at distance matrix heat-map)?

9. It is now time to _improve crossover_. Use either _line crossover_ or _box crossover_, see Unit 3 lecture slides 29 to 31. 
   Implement, test and re-run the simulation to see if results are better.
 
10. It is clear that ideal tours should not present edges that cross each other. Yet, for large enough ```N``` you will probably find that there are lots of crossings in the end result. Implement and test a new mutation operator that does the following:

     * Copies the parental array into the offspring array
     * Selects two random positions in the offspring array and inverts the order of the floats stored between those positions.
     * Returns the offspring
Does this operator help? 